# Five-day exact-Hessian compile benchmark

This standalone A100 benchmark isolates one question: does `torch.compile` become worthwhile for the original five-day exact-Hessian collocation problem?

It compares four CUDA arms using the corrected, vmap-safe scaling-and-squaring matrix exponential:

- **eager_ss**: eager exact Hessian.
- **compiled_ss**: default Inductor compilation and Triton fusion.
- **cudagraphs_ss**: Dynamo/AOTAutograd decomposition followed by CUDA Graph capture, without Inductor code generation.
- **cuda_graph_ss**: direct `torch.cuda.CUDAGraph` capture of the eager tensor-only Hessian, with no Dynamo/AOT tracing. Replay is checked against eager at both the capture inputs and a probe that perturbs every dynamic input buffer.

All arms use the same five-day workload and stop after 20 IPOPT iterations. First-call and warmed timings are reported separately, together with CPU count and PyTorch's compilation phase timings. This is a dispatch benchmark, not a convergence comparison. Native compilation is excluded because PyTorch's generated higher-order `matrix_exp` graph fails inside Inductor on CUDA.

The final cell also compares Inductor against the previously measured 24-hour timings to show scaling of cold compilation and warmed Hessian execution.

In [ ]:
import json
import os
from pathlib import Path
import platform
import subprocess
import sys

import torch

REPO_URL = "https://github.com/JBjoernskov/Twin4Build.git"
CANDIDATE_REF = "feature/issue-126/reduce-hessian-dispatch"
ROOT = Path("/content/twin4build_full_compile_hessian")
CHECKOUT = ROOT / "candidate"

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA Colab runtime is required.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", f"git+{REPO_URL}@{CANDIDATE_REF}"],
    check=True,
)
ROOT.mkdir(parents=True, exist_ok=True)
if CHECKOUT.exists():
    subprocess.run(["git", "fetch", "--quiet", "origin", CANDIDATE_REF], cwd=CHECKOUT, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{CANDIDATE_REF}"], cwd=CHECKOUT, check=True)
else:
    subprocess.run(
        ["git", "clone", "--quiet", "--depth", "1", "--branch", CANDIDATE_REF, REPO_URL, str(CHECKOUT)],
        check=True,
    )

ss_source = (CHECKOUT / "twin4build/systems/utils/discrete_statespace_system.py").read_text()
transcription_source = (CHECKOUT / "twin4build/estimator/_transcription.py").read_text()
if (
    "delta = 2.0 * delta + delta @ delta" not in ss_source
    or "class _CudaGraphCallable" not in transcription_source
):
    raise RuntimeError("Candidate checkout lacks corrected SS or direct CUDA Graph capture.")

props = torch.cuda.get_device_properties(0)
print(json.dumps({
    "gpu": props.name,
    "gpu_memory_gb": props.total_memory / 1e9,
    "torch": torch.__version__,
    "python": platform.python_version(),
    "candidate_ref": CANDIDATE_REF,
}, indent=2))

In [ ]:
# Reuse the production-workload runner maintained by the focused dispatch
# benchmark, while keeping this notebook's results in an independent directory.
dispatch_notebook = json.loads(
    (CHECKOUT / "twin4build/examples/gpu_hessian_dispatch_benchmark.ipynb").read_text()
)
runner_cells = [
    "".join(cell.get("source", []))
    for cell in dispatch_notebook["cells"]
    if cell.get("cell_type") == "code"
    and "RUNNER = ROOT / \"run_hessian_arm.py\"" in "".join(cell.get("source", []))
]
if len(runner_cells) != 1:
    raise RuntimeError("Could not identify the exact-Hessian benchmark runner cell.")
exec(compile(runner_cells[0], "gpu_hessian_dispatch runner", "exec"), globals())

In [ ]:
BENCH_HOURS = 120
BENCH_MAXITER = 20
CURRENT_REF = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=CHECKOUT, text=True
).strip()
RESULT_FILES = {
    "eager_ss": ROOT / "eager_ss_fixed_120h.json",
    "compiled_ss": ROOT / "compiled_ss_fixed_120h.json",
    "cudagraphs_ss": ROOT / "cudagraphs_ss_fixed_120h.json",
    "cuda_graph_ss": ROOT / "cuda_graph_ss_fixed_120h.json",
}
COMPATIBLE_REFS = {
    "eager_ss": {"592015e", "cf122b1", "73dbc1b", "38219a5", CURRENT_REF},
    "compiled_ss": {"592015e", "cf122b1", "73dbc1b", "38219a5", CURRENT_REF},
    "cudagraphs_ss": {"cf122b1", "73dbc1b", "38219a5", CURRENT_REF},
    "cuda_graph_ss": {CURRENT_REF},
}


def reusable(path, label):
    if not path.exists():
        return False
    try:
        row = json.loads(path.read_text())
        return (
            row.get("ref") in COMPATIBLE_REFS[label]
            and row.get("arm") == label
            and row.get("hours") == BENCH_HOURS
            and row.get("maxiter") == BENCH_MAXITER
        )
    except (json.JSONDecodeError, OSError):
        return False


for label, result_file in RESULT_FILES.items():
    if reusable(result_file, label):
        print(f"Reusing current {label} result.")
        continue
    print(f"\nRunning five-day {label} exact Hessian ...", flush=True)
    env = os.environ.copy()
    env["T4B_BENCH_HOURS"] = str(BENCH_HOURS)
    env["T4B_BENCH_MAXITER"] = str(BENCH_MAXITER)
    env["TWIN4BUILD_TRANSFORM_MATRIX_EXP"] = "ss"
    env["PYTHONPATH"] = str(CHECKOUT) + os.pathsep + env.get("PYTHONPATH", "")
    completed = subprocess.run(
        [sys.executable, str(RUNNER), str(CHECKOUT), label, str(result_file)],
        cwd=CHECKOUT,
        env=env,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr, flush=True)
    if completed.returncode:
        raise RuntimeError(
            f"{label} failed with exit code {completed.returncode}.\n"
            + "\n".join(completed.stderr.splitlines()[-80:])
        )

rows = [json.loads(path.read_text()) for path in RESULT_FILES.values()]
print("\nBoth five-day compile arms completed.")

In [ ]:
import math
import pandas as pd

REFERENCE_24H = {
    "eager_first_seconds": 0.517562453,
    "eager_warmed_median_seconds": 0.392828581,
    "compiled_first_seconds": 239.165338908,
    "compiled_warmed_median_seconds": 0.051575127,
}

summary = pd.DataFrame(rows)
eager = summary.loc[summary.arm == "eager_ss"].iloc[0]
inductor = summary.loc[summary.arm == "compiled_ss"].iloc[0]
cudagraphs = summary.loc[summary.arm == "cudagraphs_ss"].iloc[0]
direct_graph = summary.loc[summary.arm == "cuda_graph_ss"].iloc[0]
summary["solver_speedup_vs_eager"] = eager.solver_seconds / summary.solver_seconds
summary["hessian_total_speedup_vs_eager"] = (
    eager.hessian_total_seconds / summary.hessian_total_seconds
)
summary["hessian_warmed_speedup_vs_eager"] = (
    eager.hessian_warmed_median_seconds / summary.hessian_warmed_median_seconds
)


def backend_metrics(row):
    overhead = max(0.0, row.hessian_first_seconds - row.hessian_warmed_median_seconds)
    saving = eager.hessian_warmed_median_seconds - row.hessian_warmed_median_seconds
    break_even = math.inf if saving <= 0 else 1.0 + overhead / saving
    return overhead, break_even


pd.set_option("display.max_columns", None)
display(summary.drop(columns=["compile_times"], errors="ignore"))

for name, row in (
    ("Inductor", inductor),
    ("Dynamo CUDA Graphs", cudagraphs),
    ("Direct CUDA Graph", direct_graph),
):
    overhead, break_even = backend_metrics(row)
    print(
        f"{name}:\n"
        f"  warmed Hessian speedup: "
        f"{eager.hessian_warmed_median_seconds / row.hessian_warmed_median_seconds:.3f}x\n"
        f"  total Hessian speedup (includes first call): "
        f"{eager.hessian_total_seconds / row.hessian_total_seconds:.3f}x\n"
        f"  approximate startup overhead: {overhead:.1f} s\n"
        f"  estimated break-even Hessian calls: {break_even:.0f}\n"
    )

print(
    "Inductor scaling from 24 h to 120 h:\n"
    f"  eager warmed Hessian: "
    f"{eager.hessian_warmed_median_seconds / REFERENCE_24H['eager_warmed_median_seconds']:.3f}x\n"
    f"  compiled warmed Hessian: "
    f"{inductor.hessian_warmed_median_seconds / REFERENCE_24H['compiled_warmed_median_seconds']:.3f}x\n"
    f"  compiled first call: "
    f"{inductor.hessian_first_seconds / REFERENCE_24H['compiled_first_seconds']:.3f}x"
)

for name, row in (
    ("Inductor", inductor),
    ("Dynamo CUDA Graphs", cudagraphs),
    ("Direct CUDA Graph", direct_graph),
):
    print(f"\n{name} compile phase report (CPU count={row.get('cpu_count', 'unknown')}):")
    print(row.get("compile_times", "not recorded"))
    if eager.status != row.status:
        print("WARNING: solver statuses differ.")
    objective_scale = max(1.0, abs(eager.objective))
    if abs(eager.objective - row.objective) > 1e-3 * objective_scale:
        print("WARNING: objective differs by more than 0.1% after 20 iterations.")
    if abs(eager.max_defect - row.max_defect) > 5e-3:
        print("WARNING: continuity defect differs by more than 5e-3.")